# MLflow for Traditional ML
In this notebook, we will begin to make use of MLflow to support traditional ML.

## Notebook Setup

In [1]:
# Importing the necessary Python libraries
import mlflow
import mlflow.sklearn
import optuna
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

/Users/dkhundley/Documents/Repositories/mlflow-tutorial/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Loading the Iris dataset from Scikit Learn
X, y = datasets.load_iris(return_X_y = True)

In [3]:
# Splitting the data and configuring MLflow
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
 )

mlflow.set_experiment("Iris Demo")
mlflow.sklearn.autolog(silent=True)

def build_model(trial):
    return Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "classifier",
                LogisticRegression(
                    C=trial.suggest_float("C", 1e-3, 1e2, log=True),
                    solver=trial.suggest_categorical("solver", ["lbfgs", "saga"]),
                    max_iter=trial.suggest_int("max_iter", 150, 600),
                    random_state=42,
                ),
            ),
        ]
    )

2026/04/18 17:02:20 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/18 17:02:20 INFO mlflow.store.db.utils: Updating database tables
2026/04/18 17:02:20 INFO mlflow.tracking.fluent: Experiment with name 'Iris Demo' does not exist. Creating a new experiment.


In [4]:
# Optimizing hyperparameters with Optuna, then training with the best result
def objective(trial):
    model = build_model(trial)

    with mlflow.start_run(nested=True, run_name=f"optuna_trial_{trial.number}"):
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        metrics = {
            "accuracy": accuracy_score(y_test, y_pred),
            "precision_macro": precision_score(y_test, y_pred, average="macro"),
            "recall_macro": recall_score(y_test, y_pred, average="macro"),
            "f1_macro": f1_score(y_test, y_pred, average="macro"),
        }
        mlflow.log_metrics(metrics)

    return metrics["f1_macro"]

study = optuna.create_study(direction="maximize", study_name="iris_logreg_optimization")

with mlflow.start_run(run_name="optuna_hyperparameter_search"):
    study.optimize(objective, n_trials=30, show_progress_bar=False)
    mlflow.log_params({f"best_{k}": v for k, v in study.best_params.items()})
    mlflow.log_metric("best_f1_macro", study.best_value)

best_params = study.best_params

with mlflow.start_run(run_name="best_model_training"):
    final_trial = optuna.trial.FixedTrial(best_params)
    final_model = build_model(final_trial)
    final_model.fit(X_train, y_train)

    final_predictions = final_model.predict(X_test)
    final_metrics = {
        "accuracy": accuracy_score(y_test, final_predictions),
        "precision_macro": precision_score(y_test, final_predictions, average="macro"),
        "recall_macro": recall_score(y_test, final_predictions, average="macro"),
        "f1_macro": f1_score(y_test, final_predictions, average="macro"),
    }

    mlflow.log_params(best_params)
    mlflow.log_metrics(final_metrics)

print("Best hyperparameters:", best_params)
print("Final model metrics:", final_metrics)

[I 2026-04-18 17:02:20,825] A new study created in memory with name: iris_logreg_optimization
[I 2026-04-18 17:02:23,075] Trial 0 finished with value: 0.8294970161977835 and parameters: {'C': 0.023270263351225357, 'solver': 'saga', 'max_iter': 495}. Best is trial 0 with value: 0.8294970161977835.
[I 2026-04-18 17:02:24,576] Trial 1 finished with value: 0.9665831244778612 and parameters: {'C': 3.7305694596981827, 'solver': 'lbfgs', 'max_iter': 446}. Best is trial 1 with value: 0.9665831244778612.
/Users/dkhundley/Documents/Repositories/mlflow-tutorial/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
[I 2026-04-18 17:02:26,072] Trial 2 finished with value: 1.0 and parameters: {'C': 51.68783985715131, 'solver': 'saga', 'max_iter': 166}. Best is trial 2 with value: 1.0.
[I 2026-04-18 17:02:27,563] Trial 3 finished with value: 0.9333333333333332 and parameters: {'C': 0.88

Best hyperparameters: {'C': 51.68783985715131, 'solver': 'saga', 'max_iter': 166}
Final model metrics: {'accuracy': 1.0, 'precision_macro': 1.0, 'recall_macro': 1.0, 'f1_macro': 1.0}
